# Wildfire MARL — Final all-phases Colab (parallel)

End-to-end pipeline for the AAAI submission, **built to run studies in parallel to save wall-clock time**:

| Phase | What | Parallelism |
|---|---|---|
| **3** | **Reused** — the frozen 5-seed checkpoints are the source of truth; optionally re-emit tables | none (no retraining) |
| **4** | Component ablations (w/o comms / hierarchy / learned-tactical / RL / shaping) + easy/med/hard robustness | `run_phase4_ablations.py --parallel` (30 trainings across cores; `full` reuses the frozen model) |
| **5** | GIS rollout GIFs (4 policies x 2 regions) | rendered in the parallel fan-out |
| **6** | Cross-region transfer matrix + failure modes + transfer GIFs | rendered in the parallel fan-out |

**Design.** This notebook **reuses your frozen Phase-3 checkpoints** (a guard cell verifies they are
present in Drive and refuses to silently retrain). The one training-heavy step, Phase-4 ablations,
parallelizes *internally* across cores. Every other study is **eval-only over the frozen checkpoints**
and independent — so Phases 5, 6, and the Phase-4 robustness sweep launch **concurrently** in a single
fan-out cell. Wall-clock is `P4_ablation_train + max(eval studies)` instead of the sum.

**Prerequisite.** Upload your local `wildfire_phase3_multiseed/` (the 30 `.pt` files + `FREEZE.json`)
to `MyDrive/wildfire_phase3_multiseed` — they are gitignored, so the clone does not include them.

**Runtime.** T4 GPU + High-RAM. Idempotent & Drive-backed: re-run any cell if the session drops.
Set `REPO_URL`/`BRANCH` below to your pushed branch first.

In [ ]:
# 0. Sanity: GPU + CPU cores + RAM
!nvidia-smi -L
import os, psutil
print(f"CPU cores: {os.cpu_count()}  |  RAM: {psutil.virtual_memory().total/1e9:.1f} GB")

In [ ]:
# 1. Config — EDIT THESE
import os
REPO_URL    = "https://github.com/aliakarma/wildfire-rl.git"
BRANCH      = "additional"        # branch with all phases pushed
TRAIN_STEPS = 100000              # per learned method (lower to ~60000 if short on time)
EPISODES    = 15                  # eval episodes per seed group (Phase-3 main table)
EVAL_EPISODES = 10                 # episodes for the secondary studies (transfer/failure/robustness);
                                   # lower = proportionally faster. Raise to 15 for full parity.
REGIONS     = "saudi,california"
SEEDS       = "42,1042,2042,3042,4042"   # training + eval seed groups
CKPT_DIR    = "wildfire_phase3_multiseed"  # frozen dir name (matches the shipped results)
# Concurrency for the training sweeps AND the eval fan-out. ~1 core per job; lower on RAM pressure.
PARALLEL    = min(6, (os.cpu_count() or 2))
print("PARALLEL =", PARALLEL)

In [ ]:
# 2. Mount Drive (persists checkpoints/results across sessions -> resumable)
from google.colab import drive
drive.mount('/content/drive')
DRIVE   = '/content/drive/MyDrive'
OUT     = f'{DRIVE}/wildfire_phase3'          # single-seed / smoke
OUT_MS  = f'{OUT}_multiseed'                   # frozen 5-seed checkpoints (Phase 3) -> feeds 4/5/6
P4_OUT  = f'{DRIVE}/wildfire_phase4'           # ablations + robustness
P6_OUT  = f'{DRIVE}/wildfire_phase6'           # transfer + failure modes
GIF5    = f'{DRIVE}/wildfire_phase5_gifs'
GIF6    = f'{DRIVE}/wildfire_phase6_gifs'
import os
for d in (OUT, OUT_MS, P4_OUT, P6_OUT, GIF5, GIF6):
    os.makedirs(d, exist_ok=True)
print('Frozen checkpoints ->', OUT_MS)

In [ ]:
# 3. Clone (or update) the repo
import os
if not os.path.exists('/content/wildfire-rl'):
    !git clone --depth 1 --branch {BRANCH} {REPO_URL} /content/wildfire-rl
else:
    !cd /content/wildfire-rl && git fetch --depth 1 origin {BRANCH} && git checkout {BRANCH} && git reset --hard origin/{BRANCH}
%cd /content/wildfire-rl
!git log --oneline -1

In [ ]:
# 4. Build the Cell2Fire binary (skips if it already runs)
import subprocess, os
BIN = 'third_party/firehose/cell2fire/Cell2FireC/Cell2Fire'
def runs():
    try:
        return subprocess.run([f'./{BIN}', '--help'], capture_output=True, timeout=10).returncode is not None and os.path.exists(BIN)
    except Exception:
        return False
if not runs():
    !sudo apt-get -qq update && sudo apt-get -qq install -y libboost-all-dev libeigen3-dev >/dev/null
    !cd third_party/firehose/cell2fire/Cell2FireC && make -f Makefile_UBUNTU EIGENDIR=/usr/include/eigen3/ 2>&1 | tail -3
!ls -la {BIN} && file {BIN}

In [ ]:
# 5. Install the package (keep Colab's CUDA torch; add only what's missing)
!pip install -q gymnasium==1.0.0 pettingzoo==1.26.1 contextily xyzservices
!pip install -q -e . --no-deps
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
import wildfire_marl; print('wildfire_marl OK')
import contextily; print('contextily', contextily.__version__)

In [ ]:
# 6. Reusable parallel runner — launches shell commands concurrently (throttled), streams a summary.
import subprocess, time, os
def run_parallel(jobs: dict, max_concurrent: int = 4, logdir: str = '/content/logs'):
    """jobs = {name: shell_command}. Runs up to max_concurrent at once; blocks until all finish.
    Per-job output -> {logdir}/{name}.log. Returns {name: returncode}."""
    os.makedirs(logdir, exist_ok=True)
    pending, running, codes = list(jobs.items()), {}, {}
    t0 = time.time()
    while pending or running:
        while pending and len(running) < max_concurrent:
            name, cmd = pending.pop(0)
            lf = open(f'{logdir}/{name}.log', 'w')
            p = subprocess.Popen(cmd, shell=True, stdout=lf, stderr=subprocess.STDOUT,
                                 cwd='/content/wildfire-rl')
            running[name] = (p, lf)
            print(f'[START {time.time()-t0:6.0f}s] {name}  (active {len(running)})', flush=True)
        time.sleep(5)
        for name in list(running):
            p, lf = running[name]
            if p.poll() is not None:
                lf.close(); codes[name] = p.returncode
                tag = 'DONE' if p.returncode == 0 else f'FAIL({p.returncode})'
                print(f'[{tag} {time.time()-t0:6.0f}s] {name}   (last log lines below)', flush=True)
                print('   ' + '   '.join(open(f'{logdir}/{name}.log').read().splitlines()[-3:][-3:]), flush=True)
                del running[name]
    print('\nSummary:', {k: ('ok' if v == 0 else 'FAIL') for k, v in codes.items()}, flush=True)
    return codes

## Guard — reuse the frozen Phase-3 checkpoints (no silent retrain)

The 30 frozen `.pt` weights are **gitignored**, so the repo clone does *not* contain them — they must
already sit in `OUT_MS` on your Drive (from your original Phase-3 run). This cell **fails loudly**
with upload instructions if any are missing, so Phases 4/5/6 always build on your paper's frozen
`da4a381d` results rather than accidentally retrained ones.

In [ ]:
# 7. GUARD: verify the frozen checkpoints are present (do NOT silently retrain).
import os, glob, json
EXPECTED = [(m, r, s) for m in ('mappo', 'commnet', 'hiercomm_heur')
                      for r in REGIONS.split(',') for s in SEEDS.split(',')]
missing = [f'checkpoint_{m}_{r}_s{s}.pt' for m, r, s in EXPECTED
           if not os.path.exists(f'{OUT_MS}/checkpoint_{m}_{r}_s{s}.pt')]
have = len(glob.glob(f'{OUT_MS}/*.pt'))
if missing:
    raise SystemExit(
        f"\n[STOP] {len(missing)}/{len(EXPECTED)} frozen checkpoints missing from:\n  {OUT_MS}\n"
        f"  (found {have} .pt there). These weights are gitignored, so the clone lacks them.\n\n"
        f"  -> Upload your local `wildfire_phase3_multiseed/` (the 30 .pt files + FREEZE.json)\n"
        f"     to that Drive folder, then re-run this cell.\n"
        f"     e.g. still missing: {missing[:3]}\n\n"
        f"  (To instead RETRAIN from scratch, replace this cell with the Phase-3 sweep from\n"
        f"   notebooks/phase3_colab.ipynb.)")
fp = (json.load(open(f'{OUT_MS}/FREEZE.json'))['fingerprint_sha256'][:8]
      if os.path.exists(f'{OUT_MS}/FREEZE.json') else 'n/a')
print(f"[OK] all {len(EXPECTED)} frozen checkpoints present in {OUT_MS}  (fingerprint {fp})")
print("     Phases 4/5/6 will run on these weights. Phase-3 retraining is skipped.")

In [ ]:
# 8. (Optional) reproduce the Phase-3 tables from the frozen checkpoints — SKIPS training entirely
# (every checkpoint already exists, so run_phase3 only re-evaluates). Safe: deterministic re-eval
# reproduces phase3_summary.json / phase3_main_table.tex. Skip this cell if you trust the frozen tables.
!python scripts/run_phase3.py \
    --methods noop,value_first,greedy_risk,local_reactive,mappo,commnet,hiercomm_heur \
    --regions {REGIONS} --train-seeds {SEEDS} --parallel {PARALLEL} \
    --train-steps {TRAIN_STEPS} --episodes {EPISODES} --out {OUT_MS}
print(open(f'{OUT_MS}/phase3_main_table.tex').read())

## Phase 4 — Component ablations (parallel training) + regime robustness

Retrains the proposed model with one component removed at a time (`wo_comms`, `wo_shaping`,
`wo_rl_finetune`; plus eval-only `full`=frozen model, `wo_hierarchy`=CommNet, `wo_learned_tactic`
=Chebyshev). Only **30 trainings** (3 variants x 2 regions x 5 seeds; `wo_rl_finetune` is BC-only
and fast) run **in parallel across cores** — the `full` reference reuses the frozen `hiercomm_heur`
checkpoints, so it is byte-identical to the shipped model and costs nothing. Robustness is eval-only
and runs later in the fan-out.

In [ ]:
# 9. Phase-4 ablation training (parallel) + eval. Long pole; idempotent (skips existing checkpoints).
!python scripts/run_phase4_ablations.py --study ablation \
    --ckpt-dir {OUT_MS} --out {P4_OUT} --parallel {PARALLEL} \
    --train-seeds {SEEDS} --eval-seeds {SEEDS} --episodes {EPISODES} --train-steps {TRAIN_STEPS}
print(open(f'{P4_OUT}/phase4_ablation_table.tex').read())

## Phases 4b/6 — PARALLEL stats fan-out (resumable)

The three statistical studies below are **eval-only over `OUT_MS`** and independent, so they run
concurrently: Phase-4 regime robustness, Phase-6 transfer matrix, Phase-6 failure modes. They are
**resumable** — each writes completed (policy × region × condition) units to a CSV on Drive as it
goes and **skips them on re-run**, so a Colab disconnect only costs the single in-flight unit. Just
re-run this cell to continue. GIFs are rendered *afterward* (next cell) so the paper-critical numbers
land first.

**Timing.** On a slow/contended Colab CPU these ~3.5k rollouts can take **1–3 h**. Lower
`EVAL_EPISODES` (cell 2) to trade a little statistical power for proportional speed.

In [ ]:
# 10. Stats fan-out — 3 concurrent, resumable studies (re-run to continue after a disconnect).
jobs = {
 'p4_robustness':     f'python scripts/run_phase4_ablations.py --study robustness --ckpt-dir {OUT_MS} '
                      f'--out {P4_OUT} --train-seeds {SEEDS} --eval-seeds {SEEDS} --episodes {EVAL_EPISODES}',
 'p6_transfer':       f'python scripts/run_phase6_transfer.py --study transfer --ckpt-dir {OUT_MS} '
                      f'--out {P6_OUT} --train-seeds {SEEDS} --eval-seeds {SEEDS} --episodes {EVAL_EPISODES}',
 'p6_failure_modes':  f'python scripts/run_phase6_transfer.py --study generalization --ckpt-dir {OUT_MS} '
                      f'--out {P6_OUT} --train-seeds {SEEDS} --eval-seeds {SEEDS} --episodes {EVAL_EPISODES}',
}
codes = run_parallel(jobs, max_concurrent=min(PARALLEL, 3))
assert all(v == 0 for v in codes.values()), f'some studies failed: {codes} (see /content/logs/*.log)'

## Phases 5/6 — rollout GIFs (render last)

GIF rendering is the slowest per-item work (150 basemap frames per policy), so it runs *after* the
stats. Re-running re-renders (fast to skip by just not running this cell once the GIFs exist).

In [ ]:
# 10b. Prefetch tiles once, then render Phase-5 + Phase-6 GIFs concurrently.
!python scripts/prefetch_basemaps.py --regions {REGIONS} --styles EsriWorldTopo
gif_jobs = {
 'p5_gifs': f'python scripts/render_phase5_gifs.py --ckpt-dir {OUT_MS} --out {GIF5} '
            f'--seed 42 --regions {REGIONS} --max-steps 150 --style EsriWorldTopo',
 'p6_gifs': f'python scripts/render_phase6_transfer.py --ckpt-dir {OUT_MS} --out {GIF6} '
            f'--seed 42 --regions {REGIONS} --max-steps 150',
}
codes = run_parallel(gif_jobs, max_concurrent=min(PARALLEL, 2))
assert all(v == 0 for v in codes.values()), f'GIF render failed: {codes} (see /content/logs/*.log)'

## Results — tables, transfer matrix, failure modes, GIFs

In [ ]:
# 11. Print every LaTeX table + key stats
import json, os
def show(path, title):
    if os.path.exists(path):
        print(f'\n===== {title} =====\n' + open(path).read())
    else:
        print(f'(missing: {path})')
show(f'{OUT_MS}/phase3_main_table.tex',        'Phase 3 — main results')
show(f'{P4_OUT}/phase4_ablation_table.tex',    'Phase 4 — component ablation')
show(f'{P6_OUT}/phase6_transfer_table.tex',    'Phase 6 — cross-region transfer')
show(f'{P6_OUT}/phase6_generalization_table.tex', 'Phase 6 — failure modes (dWEL)')

# Phase-4 robustness monotonicity + Phase-6 transfer asymmetry
rob = f'{P4_OUT}/robustness_summary.json'
if os.path.exists(rob):
    s = json.load(open(rob)); regimes = s['protocol']['regimes']
    print('\n===== Phase 4 — regime robustness (No-Op WEL should rise with difficulty) =====')
    for region, reg in s['regions'].items():
        noop = [reg.get(r, {}).get('noop', {}).get('WEL_mean', float('nan')) for r in regimes]
        hc   = [reg.get(r, {}).get('hiercomm_heur', {}).get('WEL_mean', float('nan')) for r in regimes]
        print(f'  {region:11s} No-Op WEL {[round(x,1) for x in noop]}  |  HierComm WEL {[round(x,1) for x in hc]}')
tr = f'{P6_OUT}/transfer_summary.json'
if os.path.exists(tr):
    s = json.load(open(tr))
    print('\n===== Phase 6 — transfer robustness (TRS = transfer/native ISR) =====')
    for kind, e in s['policies'].items():
        for d, v in e['directions'].items():
            print(f"  {kind:14s} {d:22s} TRS={v['TRS_ISR']:.2f}  (native {v['native_ISR']:.2f} -> transfer {v['transfer_ISR']:.2f})")

In [ ]:
# 12. Display key GIFs + snapshots inline
from IPython.display import Image as IPImage, display
import os
for r in REGIONS.split(','):
    for path, cap in [(f'{GIF5}/comparison_grid_{r}.gif',   f'Phase 5 — {r} 4-policy comparison'),
                      (f'{GIF6}/fig_transfer_hiercomm_heur_{r}.png', f'Phase 6 — {r} native vs transferred')]:
        if os.path.exists(path):
            print(f'\n=== {cap} ==='); display(IPImage(filename=path, width=900 if path.endswith('png') else None))

In [ ]:
# 13. (Optional) zip all tables/figures (excludes .pt) for download
ZIP = '/content/wildfire_final_artifacts.zip'
!cd {DRIVE} && zip -qr {ZIP} wildfire_phase3_multiseed wildfire_phase4 wildfire_phase6 \
    wildfire_phase5_gifs wildfire_phase6_gifs -x '*.pt' '*/logs/*' '*/envmods/*'
from google.colab import files; files.download(ZIP)